<a href="https://colab.research.google.com/github/calvinWANG-3333/big-data/blob/main/colab_assignment_1_2026_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1><center>Big Data Algorithms Techniques & Platforms</center></h1>
<h2>
<hr style=" border:none; height:3px;">
<center>Spark and MapReduce</center>
<hr style=" border:none; height:3px;">
</h2>

# Rules and Environment set-up

<p align="justify">
<font size="3">
The assigment includes 22 questions on PySpark RDD MapReduce. For this assignment use **ONLY** PySpark RDD transformations (`.map()`, `.filter()`, `.reduceByKey()`, `.aggregateByKey()`, `.flatMap()`, etc.).

**DO NOT** use PySpark DataFrames, Datasets, or Spark SQL.

</font>
</p>





The Base Adresse Nationale (BAN) is the single official database containing every localized postal address across France. Managed jointly by French public authorities—including the IGN (National Institute of Geographic and Forest Information), La Poste, and local municipalities (communes)—it provides authoritative geospatial reference data for public services, emergency response teams, navigation systems, and private logistics.

The dataset contains millions of address points spanning mainland France and overseas territories. Each record represents an individual address point and links textual address metadata to exact geographic coordinates:


*   Administrative Boundaries: Standardized local codes, including 5-digit postal codes (code_postal), 5-character official INSEE municipality identifiers (code_insee), municipality names (nom_commune), and department codes (code_departement).

*   National Identifiers: Unique address identifiers (id) and FANTOIR street directory codes (id_fantoir) used for official record keeping.

*   Geographic Coordinates: Precise WGS 84 latitude and longitude coordinates mapping the location of each address.
*   Address Structure: Detailed building numbers (numero), repetition suffixes (rep such as bis or ter), and official street names (nom_voie or lieu-dit).



In [1]:
# ==========================================
# SETUP & DATA CLEANING
# ==========================================
import csv
import math
from pyspark import SparkConf, SparkContext

conf = SparkConf().setAppName("BAN_MapReduce_22_Exercises")
sc = SparkContext.getOrCreate(conf=conf)

#Example - how to load one partition of the dataset
raw_rdd = sc.textFile("adresses-france.csv.gz")
header = raw_rdd.first()

def parse_line(line):
    return list(next(csv.reader([line], delimiter=';')))

# Clean RDD: Filter out header and invalid column lengths
adresses_rdd = (
    raw_rdd.filter(lambda line: line != header)
    .map(parse_line)
    .filter(lambda cols: len(cols) >= 11)
)

adresses_rdd.cache()
print(f"Loaded & cached {adresses_rdd.count():,} address records.")


Loaded & cached 26,121,849 address records.


---
## Exercise 1: Count Addresses per Department
**Goal:** Compute address counts per department (`code_departement`). Output top 10.

**Allowed RDD Functions:** `.map()`, `.reduceByKey()`, `.sortBy()`

**There's no code department in downloaded file so I use the code_postal**

In [3]:
print("Exercise 1")
# TODO: Write your MapReduce code here
count = (
    adresses_rdd.map(lambda x : (x[5],1))
                .reduceByKey(lambda x, y: x + y)
                .sortBy(lambda x: x[1], ascending = False)
)

count.take(10)

Exercise 1


[('51100', 34445),
 ('29200', 32846),
 ('59200', 30502),
 ('33000', 29304),
 ('79000', 29209),
 ('21000', 29137),
 ('72000', 28239),
 ('97430', 28119),
 ('97410', 27141),
 ('59100', 26398)]

---
## Exercise 2: Top 10 Street Names Nationally
**Goal:** Find top 10 most common street names (`nom_voie`), uppercase, ignoring empty names.

**Allowed RDD Functions:** `.map()`, `.filter()`, `.reduceByKey()`, `.takeOrdered()`

In [4]:
print("Exercise 2")
# TODO: Write your MapReduce code here

top_10_street = (
    adresses_rdd.filter(lambda x: x[4] != "")
                .map(lambda x : (x[4].upper(),1))
                .reduceByKey(lambda x, y: x + y)
                .takeOrdered(10,key= lambda x : -x[1])
)

top_10_street


Exercise 2


[('GRANDE RUE', 155854),
 ('RUE PRINCIPALE', 83678),
 ("RUE DE L'EGLISE", 58466),
 ('RUE DU MOULIN', 55709),
 ('RUE DE LA GARE', 55645),
 ('RUE PASTEUR', 54928),
 ('RUE DE LA RÉPUBLIQUE', 52560),
 ('RUE VICTOR HUGO', 50844),
 ('RUE DE LA MAIRIE', 48496),
 ('RUE JEAN JAURÈS', 42202)]

notice that compared to rdd.sortBy the rdd.takeOrder() is actually an action, it directly return a python list rather than a rdd

---
## Exercise 3: Geographical Center per Department
**Goal:** Calculate average latitude and longitude per department using a single map-reduce step (combiner pattern).

**Allowed RDD Functions:** `.map()`, `.filter()`, `.reduceByKey()` / `.aggregateByKey()`, `.mapValues()`

In [ ]:
print("Exercise 3")
# TODO: Write your MapReduce code here
result = (
    adresses_rdd.filter(lambda x : x[5] != "")
                .map(lambda x: (x[5],(float(x[12]),float(x[13]),1)))
                .reduceByKey(lambda x, y: (x[0] + y[0] ,x[1] +y[1] ,x[2] +y[2]))
                .mapValues(lambda x: ( x[0] / x[2], x[1] / x[2]))
                )

result.take(2)


Exercise 3


---
## Exercise 4: Municipality with Most Unique Streets
**Goal:** Find the municipality (`code_insee`, `nom_commune`) with the largest count of DISTINCT street names.

**Allowed RDD Functions:** `.map()`, `.distinct()`, `.reduceByKey()`, `.first()` / `.sortBy()`

upon reading the documentation the .distinct() is a wide transform and for a rdd full of tuple, that only every element within the tuple is the same would be considered as same

In [11]:
print("Exercise 4")
# TODO: Write your MapReduce code here

result = (
    adresses_rdd.map(lambda x : ((x[6],x[7]),x[4]))
                .distinct()
                .map(lambda x: (x[0],1))
                .reduceByKey(lambda x, y: x+y)
                .sortBy(lambda x: x[1], ascending=False)
                .first()
)

result


Exercise 4


(('01053', 'Bourg-en-Bresse'), 617)

---
## Exercise 5: Suffix Repetitions in Postal Code 75001
**Goal:** Extract addresses in postal code `75001` with repetition suffixes (`bis`, `ter`). Sort them by numeric street number.

**Allowed RDD Functions:** `.filter()`, `.map()`, `.groupByKey()`, `.mapValues()`

In [ ]:
print("Exercise 5")
# TODO: Write your MapReduce code here


---
## Exercise 6: Bounding Box per Department
**Goal:** For each department, determine the minimum and maximum lat and lon: `(min_lat, max_lat, min_lon, max_lon)`.

**Allowed RDD Functions:** `.map()`, `.filter()`, `.reduceByKey()`

In [ ]:
print("Exercise 6")
# TODO: Write your MapReduce code here


---
## Exercise 7: Longest Street Name by Character Count
**Goal:** Find the street address record that has the longest street name length.

**Allowed RDD Functions:** `.map()`, `.reduce()` or `.takeOrdered()`

In [ ]:
print("Exercise 7")
# TODO: Write your MapReduce code here


---
## Exercise 8: Histogram of Street Numbers
**Goal:** Group street numbers into buckets (`1-10`, `11-50`, `51-100`, `101-500`, `501+`) and count addresses in each bucket.

**Allowed RDD Functions:** `.map()`, `.reduceByKey()`

In [ ]:
print("Exercise 8")
# TODO: Write your MapReduce code here


---
## Exercise 9: Municipalities Sharing the Same Street Name
**Goal:** Find street names that exist in more than 50 distinct municipalities.

**Allowed RDD Functions:** `.map()`, `.distinct()`, `.reduceByKey()`, `.filter()`

In [ ]:
print("Exercise 9")
# TODO: Write your MapReduce code here


---
## Exercise 10: Department Address Density Rank
**Goal:** Count unique municipalities per department and calculate the ratio: `total_addresses / total_municipalities`. Display top 5 departments by this ratio.

**Allowed RDD Functions:** `.map()`, `.reduceByKey()`, `.join()` / `.aggregateByKey()`

In [ ]:
print("Exercise 10")
# TODO: Write your MapReduce code here


---
## Exercise 11: Distance to Paris Zero Point
**Goal:** Compute the Haversine distance (in km) from each address to Notre-Dame Paris (48.8534 N, 2.3488 E). Find the furthest single address in mainland France (departments starting with 0-9, excluding DOM-TOM > 97).

**Allowed RDD Functions:** `.map()`, `.filter()`, `.reduce()` / `.top()`

In [ ]:
print("Exercise 11")
# TODO: Write your MapReduce code here


---
## Exercise 12: Identify Outlier Addresses
**Goal:** Find addresses whose coordinates fall outside valid French mainland/overseas boundaries (`lat < -90` or `lat > 90` or `lon < -180` or `lon > 180` or `lat == 0.0` or `lon == 0.0`). Count total outliers.

**Allowed RDD Functions:** `.filter()`, `.count()`

In [ ]:
print("Exercise 12")
# TODO: Write your MapReduce code here


---
## Exercise 13: Top Street Name per Department
**Goal:** Find the single most frequent street name for EACH department.

**Allowed RDD Functions:** `.map()`, `.reduceByKey()`, `.groupByKey()` / `.reduceByKey()`

In [ ]:
print("Exercise 13")
# TODO: Write your MapReduce code here


---
## Exercise 14: Spatial Grid Indexing
**Goal:** Partition coordinates into a spatial grid of size 0.1 x 0.1 degrees. Find the top 3 grid cells containing the highest density of addresses.
Grid cell key format: `(floor(lat / 0.1), floor(lon / 0.1))`

**Allowed RDD Functions:** `.map()`, `.reduceByKey()`, `.takeOrdered()`

In [ ]:
print("Exercise 14")
# TODO: Write your MapReduce code here


---
## Exercise 15: Find Homonym Communes
**Goal:** Identify municipality names (`nom_commune`) that correspond to multiple INSEE codes. List the top 5 commune names with the highest number of distinct INSEE codes.

**Allowed RDD Functions:** `.map()`, `.distinct()`, `.reduceByKey()`, `.filter()`, `.sortBy()`

In [ ]:
print("Exercise 15")
# TODO: Write your MapReduce code here


---
## Exercise 16: Jaccard Similarity of Street Names Between Two Departments
**Goal:** Compute the Jaccard Similarity index (size of intersection / size of union) between the set of unique street names in department `'75'` (Paris) and department `'92'` (Hauts-de-Seine).

**Allowed RDD Functions:** `.filter()`, `.map()`, `.distinct()`, RDD set operations (`.intersection()`, `.union()`) or MapReduce equivalent.

In [ ]:
print("Exercise 16")
# TODO: Write your MapReduce code here


---
## Exercise 17: Standard Deviation & Percentile Histogram Estimation
**Goal:** Compute the exact Standard Deviation and estimated 25th, 50th (Median), and 75th percentiles of street numbers nationally in a memory-efficient MapReduce pass.

**Allowed RDD Functions:** `.flatMap()`, `.aggregateByKey()` / `.reduceByKey()`, `.sortByKey()`

In [ ]:
print("Exercise 17")
# TODO: Write your MapReduce code here


---
## Exercise 18: Spatial Self-Join / Nearest Address Pair
**Goal:** Partition addresses in postal code `'75001'` into 0.005x0.005 degree spatial buckets. For each bucket, find the single pair of distinct addresses that are closest to each other in meters. Output the minimum-distance pair and the distance between them in metres per bucket.

**Note:**
- 1 degree of latitude is approximately $111.1$ km.
- 1 degree of longitude is approximately $111.1 \times \cos(\text{latitude})$ km.

**Allowed RDD Functions:** `.filter()`, `.map()`, `.reduce()`, `.aggregateByKey()`, `.sortBy()`, `.flatMap()`

In [ ]:
print("Exercise 18")
# TODO: Write your MapReduce code here


---
## Exercise 19: Street Name Bigram Analysis
**Goal:** Tokenize all street names into 2-grams (bigrams of consecutive words). Find the top 10 most frequent street name bigrams in France.

**Allowed RDD Functions:** `.flatMap()`, `.reduceByKey()`, `.takeOrdered()`

In [ ]:
print("Exercise 19")
# TODO: Write your MapReduce code here


---
## Exercise 20: Prefix Trie Frequency Analysis
**Goal:** For each letter `'A'` through `'Z'`, find the 3 most common 3-character prefixes of street names starting with that letter.

**Allowed RDD Functions:** `.flatMap()`, `.reduceByKey()`, `.groupByKey()`, `.mapValues()`

In [ ]:
print("Exercise 20")
# TODO: Write your MapReduce code here


---
## Exercise 21: Department with Maximum Homonym Communes
**Goal:** Identify the department that contains the highest count of homonym communes using two different map-reduce approaches:
- Using some intermediate result obtained from Exercise 15
- Any other approach

Identify the map and reduce stages for both solutions, and show which approach is more efficient with empirical justifications.

**Allowed RDD Functions:** `.map()`, `.distinct()`, `.reduceByKey()`, `.filter()`, `.sortBy()`

In [ ]:
print("Exercise 21")
# TODO: Write your MapReduce code here


---
## Exercise 22: Identify the Most Prominent Geographical Area
**Goal:** Identify the geographical area using bounding boxes of area 100 square kilometers containing the maximum number of addresses. How many postal codes does this geographical area share? You need to use two different map-reduce approaches:
- Without using `.reduce()`
- Without using `.reduceByKey()`

Identify the map and reduce stages for both solutions, and show which approach is more efficient with empirical justifications.

**Note:**
- 1 degree of latitude is approximately $111.1$ km.
- 1 degree of longitude is approximately $111.1 \times \cos(\text{latitude})$ km.

In [ ]:
print("Exercise 22")
# TODO: Write your MapReduce code here

sc.stop()